# M1 — LoRA fine-tuning of Qwen3-1.7B for exercise field extraction

**SI4006 · Entrega M1 — Fine-tuning baseline del proyecto**

This notebook teaches a small language model to read an exercise description and
return its catalog fields as JSON:

> `Name: cable incline pushdown` → `{"target": "lats", "equipment": "cable"}`

It runs end to end on a **free Colab T4**. Sections:

1. Setup
2. Base model and tokenizer — and why this model
3. Dataset preparation
4. Baselines (rule-based, and the same model *without* fine-tuning)
5. LoRA configuration and training
6. Final evaluation against the baselines
7. Qualitative examples and an honest reading

Everything is seeded (`seed = 42`). Cells are meant to be run in order, top to bottom.

## 1 · Setup

Clones the project (with the dataset submodule) when running on Colab, and installs
the training stack on top of whatever torch Colab already ships.

In [1]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/iamcroody/models-for-exercises-dataset.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not Path("models-for-exercises-dataset").exists():
        subprocess.run(
            ["git", "clone", "--recurse-submodules", REPO_URL], check=True
        )
    os.chdir("models-for-exercises-dataset")
    # torch comes with the Colab image; only the training stack is missing.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "transformers>=5.0", "trl>=1.9", "peft>=0.20", "datasets", "scikit-learn"],
        check=True,
    )
else:
    # Local run: notebooks/ lives one level below the repo root.
    if Path.cwd().name == "notebooks":
        os.chdir("..")

sys.path.insert(0, str(Path("scripts").resolve()))
print("working directory:", Path.cwd())

working directory: /home/jayoungh/septimoSemestre/EAFIT/AI/models-for-exercises-dataset


In [2]:
import torch

import exlib

device, DTYPE = exlib.pick_device_dtype()
BF16 = str(DTYPE).endswith("bfloat16")

print(f"torch    {torch.__version__}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"gpu      {props.name} ({props.total_memory / 1e9:.1f} GB, sm_{props.major}{props.minor})")
print(f"dtype    {DTYPE}")
print()
print(
    "Precision is picked from the device, not hard-coded. Colab's free T4 is Turing\n"
    "(sm_75) and has no bf16 units, while TRL's SFTConfig defaults bf16=True whenever\n"
    "fp16 is unset — so a hard-coded config crashes on exactly the GPU this notebook\n"
    "is required to run on. FlashAttention-2 is Ampere+ only and is left off for the\n"
    "same reason."
)

torch    2.13.0+cu130
gpu      NVIDIA GeForce RTX 5060 Ti (16.6 GB, sm_120)
dtype    torch.bfloat16

Precision is picked from the device, not hard-coded. Colab's free T4 is Turing
(sm_75) and has no bf16 units, while TRL's SFTConfig defaults bf16=True whenever
fp16 is unset — so a hard-coded config crashes on exactly the GPU this notebook
is required to run on. FlashAttention-2 is Ampere+ only and is left off for the
same reason.


## 2 · Base model and tokenizer

**Family: decoder.** Not because it scores best in isolation — a `DeBERTa-v3`-style
encoder classifier would very likely beat it on a closed 19-class problem — but because
the assignment states this model carries into M2 (RAG on top of it) and M3 (a visual
component). An encoder classifier can host neither. Choosing the marginally weaker
architecture that survives two more modules is the cheaper decision.

**Model: `Qwen/Qwen3-1.7B`.**

- **Apache-2.0 and ungated.** No Hugging Face login, no licence click-through. Llama-3.2
  is gated, which would break the "anyone can open this notebook and run it"
  requirement outright.
- **Fits the free tier with room.** ~1.0% of parameters trainable under LoRA, ~6.7 GB
  peak VRAM against a T4's 15 GB.
- **Has a forward path.** `Qwen3-VL-2B` shares this tokenizer and chat template, so M3's
  visual component is a family swap rather than a rewrite — and the dataset already
  ships a thumbnail and an animation GIF for all 1324 exercises.

### Tokenization check (Week 3, Lab A)

Before committing to a base model, look at how it splits the vocabulary the task turns
on. That matters more than usual here, because our labels are the **output**: the model
must reproduce all 19 target and 28 equipment values verbatim, so label fragmentation is
a direct measure of how hard the output space is.

In [3]:
!{sys.executable} scripts/00_tokenizer_check.py

tokenizer                                vocab   target    equip    names  instr p95
------------------------------------------------------------------------------------
Qwen/Qwen3-1.7B                         151669     2.00     1.54     1.43        128
HuggingFaceTB/SmolLM2-1.7B-Instruct      49152     2.00     1.50     1.49        128
openai-community/gpt2                    50257     2.04     1.59     1.48        129
google-bert/bert-base-uncased            30522     1.91     1.26     1.32        127
google/flan-t5-base                      32100     2.74     1.63     1.64        145

(numbers are tokens per word — lower is better)

worst-fragmented labels for the chosen tokenizer:
  target     levator scapulae (5), cardiovascular system (4), serratus anterior (4)
  equipment  olympic barbell (5), elliptical machine (4), skierg machine (4)

wrote /home/jayoungh/septimoSemestre/EAFIT/AI/models-for-exercises-dataset/reports/tokenizer_study.json


Read honestly: **BERT's lowercase WordPiece fragments our anatomy vocabulary the least**,
and `flan-t5` fragments it worst by a wide margin. Qwen3 sits at the front of the decoder
options but does not win outright.

So this study did not pick the model — the family requirement did. What it does establish
is that Qwen3 costs us nothing *within* its family, and it rules out the encoder-decoder
option on a concrete measurement rather than on taste.

In [4]:
from transformers import AutoTokenizer

tokenizer = exlib.load_tokenizer(exlib.BASE_MODEL)
print(f"{exlib.BASE_MODEL}: vocab {len(tokenizer)}, eos {tokenizer.eos_token!r}, pad {tokenizer.pad_token!r}")

/home/jayoungh/septimoSemestre/EAFIT/AI/models-for-exercises-dataset/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Qwen/Qwen3-1.7B: vocab 151669, eos '<|im_end|>', pad '<|endoftext|>'


## 3 · Dataset

1324 exercises from a pinned git submodule. Full description, licence and known biases:
[`docs/DATASET.md`](../docs/DATASET.md).

`scripts/02_prepare_data.py` builds the splits: stratified on `target`, seed 42,
written as TRL conversational prompt/completion pairs.

In [5]:
!{sys.executable} scripts/02_prepare_data.py

moved val -> train: equipment='trap bar' unseen in train (trap bar deadlift)

train   1081 examples  ->  /home/jayoungh/septimoSemestre/EAFIT/AI/models-for-exercises-dataset/data/processed/train.jsonl
val      121 examples  ->  /home/jayoungh/septimoSemestre/EAFIT/AI/models-for-exercises-dataset/data/processed/val.jsonl
test     122 examples  ->  /home/jayoungh/septimoSemestre/EAFIT/AI/models-for-exercises-dataset/data/processed/test.jsonl

label space: target=19, equipment=28
body_part derived from target (19 targets -> 10 body parts)
wrote /home/jayoungh/septimoSemestre/EAFIT/AI/models-for-exercises-dataset/data/processed/meta.json


### The model predicts two fields, not three

The catalog also has `body_part`, but it is **not** a third prediction:

- `target → body_part` is a strict function — all 19 targets map to exactly one of 10
  body parts, no exceptions in 1324 records.
- `category` is a verbatim copy of `body_part` in every record.

So `body_part` is free whenever `target` is right. Generating it would add an accuracy
column that inflates the headline number while measuring nothing, so it is derived from a
lookup table instead. `exlib.build_body_part_map` raises if a dataset bump ever makes
`target` ambiguous, so the assumption cannot rot unnoticed.

In [6]:
meta = exlib.load_meta()
train_rows = exlib.load_split("train")
val_rows = exlib.load_split("val")

print("splits:", meta["counts"])
print("predicted:", meta["predict_fields"], "| derived:", meta["derived_field"])
print()
print("=== one training example, exactly as the model sees it ===")
print(train_rows[0]["prompt"][0]["content"])
print()
print("--- expected completion ---")
print(train_rows[0]["completion"][0]["content"])

splits: {'train': 1081, 'val': 121, 'test': 122}
predicted: ['target', 'equipment'] | derived: body_part

=== one training example, exactly as the model sees it ===
Extract the structured fields for this exercise.

Name: cable incline pushdown
Instructions: Attach a straight bar to a high pulley cable machine. Stand facing away from the machine with your feet shoulder-width apart. Grasp the bar with an overhand grip, hands slightly wider than shoulder-width apart. Lean forward slightly and keep your back straight. Pull the bar down towards your thighs by extending your elbows. Pause for a moment at the bottom, then slowly return the bar to the starting position. Repeat for the desired number of repetitions.

Reply with JSON only, choosing from these exact values:
target: abductors | abs | adductors | biceps | calves | cardiovascular system | delts | forearms | glutes | hamstrings | lats | levator scapulae | pectorals | quads | serratus anterior | spine | traps | triceps | upper back
eq

The prompt lists every valid value on purpose. The **same** prompt grades the untrained
model in section 4, and a base model that has never seen our labelling conventions would
otherwise be marked down for vocabulary it was never shown — which measures our
conventions, not the model.

One definition of that prompt exists (`exlib.build_prompt`) and all three evaluations
call it. If training and evaluation built the string separately, the reported delta would
partly measure prompt drift.

In [7]:
import statistics

lengths = [
    len(tokenizer.encode(
        tokenizer.apply_chat_template(r["prompt"] + r["completion"], tokenize=False),
        add_special_tokens=False,
    ))
    for r in train_rows
]
print(f"tokens per training example: mean {statistics.mean(lengths):.0f}, max {max(lengths)}")
print("-> max_length = 512 truncates nothing, at half the cost of the 1024 default")

tokens per training example: mean 302, max 386
-> max_length = 512 truncates nothing, at half the cost of the 1024 default


## 4 · Baselines

A single number says nothing, so there are two baselines and they measure different
things.

**(a) Rule-based** — majority class, and a substring rule that looks for the label
verbatim in the exercise name. This measures *the dataset*: how much of the task is
solvable with no model at all. Both are **fitted on train and scored on val**, exactly
like the model — a baseline that has seen the evaluation set is not a baseline.

In [8]:
!{sys.executable} scripts/01_baseline.py --split val

fitted on train (1081), scored on val (121)

strategy   field        accuracy  macro-F1  leakage
---------------------------------------------------
majority   target         13.2%     0.016    5.8%
majority   equipment      30.6%     0.031   57.0%
           joint           9.1%
substring  target         18.2%     0.064    5.8%
substring  equipment      87.6%     0.783   57.0%
           joint          18.2%

wrote /home/jayoungh/septimoSemestre/EAFIT/AI/models-for-exercises-dataset/reports/baseline.json


**(b) Zero-shot — the same base model with no fine-tuning.** This is the baseline the
assignment recommends, and the only one that isolates what LoRA actually contributed.
Same prompts, same split, same greedy decoding, no adapter.

*(A few minutes on a T4.)*

In [9]:
!{sys.executable} scripts/03_eval_zeroshot.py --split val

Qwen/Qwen3-1.7B (no adapter) on cuda / torch.bfloat16
scoring 121 val examples



Loading weights:   0%|                                 | 0/311 [00:00<?, ?it/s]

Loading weights:   8%|█▉                     | 26/311 [00:00<00:01, 258.62it/s]

Loading weights:  49%|██████████▊           | 153/311 [00:00<00:00, 850.07it/s]

Loading weights: 100%|██████████████████████| 311/311 [00:00<00:00, 951.08it/s]



JSON valid        100.0%
target             32.2%  macro-F1 0.282  in-label 70.2%
equipment          89.3%  macro-F1 0.900  in-label 98.4%
joint              30.6%

wrote /home/jayoungh/septimoSemestre/EAFIT/AI/models-for-exercises-dataset/reports/zeroshot.json


This reframes the problem. The untrained model already **beats the rule-based baseline on
both fields** and emits valid JSON every single time — it clearly reads the domain. What
it does not do is respect the closed label space: roughly a third of its `target` answers
are values we never offered. It echoes the exercise name back (`"target": "dumbbell iron
cross"`) or lands one letter off a real label (`"pectors"` for `"pectorals"`).

**Constraining output to the label space — not teaching fitness — is what the fine-tune
is for.** That is the hypothesis the rest of the notebook tests.

## 5 · LoRA configuration and training

No full fine-tuning: only low-rank adapters on the frozen base weights.

| Hyperparameter | Value | Why |
|---|---|---|
| `r` | 16 | 1081 examples over a 19+28 label space is a small, narrow target. The job is constraining output to a closed vocabulary, not installing new knowledge — that needs little capacity, and higher rank mostly buys overfitting. Tested in the ablation below rather than asserted. |
| `lora_alpha` | 32 | Holds the conventional `alpha = 2r`, so the effective scale `alpha/r` stays 2.0. Without this, changing `r` would silently change update magnitude too and confound the ablation. |
| `target_modules` | all 7 linear projections | Attention **and** MLP. Mapping "cable incline pushdown" → `{lats, cable}` is lexical-semantic, and that association lives largely in the MLP blocks; attention-only adapters can reweight what the model attends to but not what it knows a term means. |
| `lora_dropout` | 0.05 | Light regularisation on a small dataset. |
| `learning_rate` | 1e-4 | TRL's documented adapter rate, ~5× a full fine-tune's, because only the freshly-initialised low-rank matrices are learning. |
| effective batch | 16 | `per_device=2 × grad_accum=8`, fixed rather than scaled to the GPU so a local run and the Colab run take identical optimisation steps. |

In [10]:
from datasets import Dataset
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

ATTENTION = ["q_proj", "k_proj", "v_proj", "o_proj"]
MLP = ["gate_proj", "up_proj", "down_proj"]

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=ATTENTION + MLP,
)

training_args = SFTConfig(
    output_dir="outputs/notebook",
    num_train_epochs=3,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    max_length=512,
    seed=exlib.SEED,
    data_seed=exlib.SEED,
    eval_strategy="epoch",
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    model_init_kwargs={"dtype": DTYPE},
    bf16=BF16,
    fp16=not BF16,
)

# Only the two columns TRL needs — the gold labels carried alongside in the JSONL
# for evaluation would otherwise be templated into the training text.
def to_dataset(rows):
    return Dataset.from_list(
        [{"prompt": r["prompt"], "completion": r["completion"]} for r in rows]
    )

trainer = SFTTrainer(
    model=exlib.BASE_MODEL,
    args=training_args,
    train_dataset=to_dataset(train_rows),
    eval_dataset=to_dataset(val_rows),
    peft_config=peft_config,
)
trainer.model.print_trainable_parameters()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 1978.91it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading weights:   7%|▋         | 22/311 [00:00<00:01, 218.48it/s]

...

Truncating eval dataset:   0%|          | 0/121 [00:00<?, ? examples/s]

Truncating eval dataset: 100%|██████████| 121/121 [00:00<00:00, 6520.17 examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/121 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset: 100%|██████████| 121/121 [00:00<00:00, 8194.51 examples/s]

trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


TRL computes the loss on the **completion only** for prompt-completion datasets
(`completion_only_loss` defaults to `True`), so the long label-space listing in the prompt
costs nothing at training time — the model is never asked to predict it.

Now train. *(~7 min on an RTX 5060 Ti, ~25–30 min on a Colab T4.)*

In [11]:
train_result = trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.053573,0.039722,0.042711,326667.000000,0.986799
2,0.033114,0.030184,0.027288,653334.000000,0.992765
3,0.015554,0.028754,0.024795,980001.000000,0.991953


In [12]:
ADAPTER_DIR = "models/notebook-r16"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

evals = [h for h in trainer.state.log_history if "eval_loss" in h]
print("eval loss per epoch: " + " -> ".join(f"{h['eval_loss']:.4f}" for h in evals))
print(f"adapter saved to {ADAPTER_DIR}")

eval loss per epoch: 0.0397 -> 0.0302 -> 0.0288
adapter saved to models/notebook-r16


## 6 · Final evaluation, against the same validation set

Every row of the table below is produced by the same scoring function
(`exlib.score_predictions`) on the same 121 validation records, so the comparison is not
three lookalike implementations quietly disagreeing.

Metrics and why:

- **accuracy** — closed-set classification, so exact match is the natural measure.
  `target` is the headline: 19 classes, the weakest baseline, the most room.
- **macro-F1** — accuracy hides the tail, and the tail is most of the label space
  (`levator scapulae` has 2 records in the entire catalog). Macro-F1 weights a rare class
  the same as `abs`.
- **JSON valid / in-label** — reported, never repaired. These are where the untrained
  model actually loses, and hiding them would make the delta look like magic.

In [13]:
# Free the training copy before loading the merged model for generation.
del trainer
import gc

gc.collect()
torch.cuda.empty_cache()

!python scripts/05_eval_finetuned.py --split val --adapter {ADAPTER_DIR} --report-name finetuned_notebook

Qwen/Qwen3-1.7B + notebook-r16 on cuda / torch.bfloat16
scoring 121 val examples



Loading weights:   0%|                                 | 0/311 [00:00<?, ?it/s]

Loading weights:   8%|█▉                     | 26/311 [00:00<00:01, 256.75it/s]

Loading weights:  50%|██████████▉           | 155/311 [00:00<00:00, 857.55it/s]

Loading weights:  90%|██████████████████▉  | 280/311 [00:00<00:00, 1031.85it/s]

Loading weights: 100%|██████████████████████| 311/311 [00:00<00:00, 962.19it/s]



                         target    equip    joint    t-F1    e-F1    JSON
-------------------------------------------------------------------------
rule: majority           13.2%   30.6%    9.1%   0.016   0.031  100.0%
rule: substring          18.2%   87.6%   18.2%   0.064   0.783  100.0%
zero-shot (no LoRA)      32.2%   89.3%   30.6%   0.282   0.900  100.0%
fine-tuned (LoRA)        84.3%   99.2%   83.5%   0.799   0.924  100.0%

delta from fine-tuning (vs the same model, untrained):
  target     accuracy  32.2% ->  84.3%   in-label  70.2% -> 100.0%
  equipment  accuracy  89.3% ->  99.2%   in-label  98.4% -> 100.0%

qualitative examples:
  [ok  ] lever unilateral row  (typical correct extraction)
         gold {'target': 'upper back', 'equipment': 'leverage machine'}
         got  {'target': 'upper back', 'equipment': 'leverage machine'}
  [ok  ] exercise ball back extension with arms extended  (correct on a rare target class)
         gold {'target': 'spine', 'equipment': 'stability b

## 7 · Ablation — and where our reasoning was wrong

The assignment invites experimenting rather than asserting, so we tested the two
hyperparameter claims we had made. Three runs, identical apart from the variable under
test, `alpha = 2r` throughout so changing `r` does not also change the update scale.

| Config | Trainable | target acc | target macro-F1 | joint | eval loss |
|---|---|---|---|---|---|
| r=8, all linear | 8.7M | 77.7% | 0.683 | 76.9% | 0.0319 |
| **r=16, all linear** | 17.4M | **85.1%** | **0.814** | **84.3%** | **0.0279** |
| r=16, attention only | 6.4M | 77.7% | 0.679 | 76.0% | 0.0348 |

**Both of our predictions were wrong.**

We expected `r=8` to be sufficient and `r=16` to risk overfitting on 1081 examples. In
fact `r=8` **underfits**, losing 7.4 points, and no run overfitted at all — eval loss was
still falling at epoch 3 in all three.

We also expected the MLP blocks to be doing the work. The attention-only run does lose
7.4 points, which looks like confirmation — until you notice it has **fewer** trainable
parameters (6.4M) than the r=8 all-linear run (8.7M) and scores *identically* to it.
Placement is confounded with capacity here.

What the data supports is narrower: **total adapter capacity drove the result, not
placement.** Both ~6–9M configurations land on 77.7% whichever modules they adapt; only
the 17.4M one reaches 85.1%. `r=16` on all linear layers is kept because it measurably
won, not because our story about MLPs was right.

A controlled placement test would need parameter-matched runs (roughly `r=16`
attention-only against `r=6` all-linear), which we did not do.


In [14]:
RUN_ABLATION = False  # set True to reproduce; roughly triples the notebook runtime

if RUN_ABLATION:
    !{sys.executable} scripts/04_train_lora.py --r 8 --run-name r8-all-linear
    !{sys.executable} scripts/05_eval_finetuned.py --adapter models/r8-all-linear --report-name finetuned_r8
    !{sys.executable} scripts/04_train_lora.py --target-modules attention --run-name r16-attention
    !{sys.executable} scripts/05_eval_finetuned.py --adapter models/r16-attention --report-name finetuned_attn
else:
    import json

    for name in ["finetuned_r8", "finetuned", "finetuned_attn"]:
        path = Path("reports") / f"{name}.json"
        if path.exists():
            r = json.loads(path.read_text())
            cfg = r.get("training", {}).get("lora", {})
            modules = "all-linear" if len(cfg.get("target_modules", [])) > 4 else "attention"
            print(
                f"r={cfg.get('r', '?'):<3} {modules:<11} "
                f"target {r['fields']['target']['accuracy']:.1%}  "
                f"equipment {r['fields']['equipment']['accuracy']:.1%}  "
                f"joint {r['joint_accuracy']:.1%}"
            )

r=8   all-linear  target 77.7%  equipment 99.2%  joint 76.9%
r=16  all-linear  target 85.1%  equipment 99.2%  joint 84.3%
r=16  attention   target 77.7%  equipment 98.4%  joint 76.0%


## 8 · Honest reading

See [`README.md`](../README.md) for the committed results table and the written
conclusion, and [`docs/DATASET.md`](../docs/DATASET.md) for the dataset's known biases —
including the ~1.7% of validation records whose name/label pair also appears in train
under different wording, which biases these numbers slightly upward.

The test split (122 records) has deliberately not been touched. It is reserved for M2, so
that the rigorous evaluation there is not run on a set already used to make decisions here.